In [1]:
import pandas as pd
import numpy as np
import sys
import random

In [7]:
tissue = ['BLOOD', 'LIVER', 'LUNG', 'KIDNEY']

In [3]:
trait = pd.read_csv('/home/lucytian/data/1_Single_Cell_PRS/2_cV2F/pheno_tissue.txt', header=None)[0].tolist()

In [8]:
trait

['INI30120', 'INI50030700', 'INI1003063', 'INI20030780']

In [ ]:
tissue_trait_dict = {'INI30120': 'BLOOD', 'INI50030700': 'KIDNEY', 'INI1003063': 'LUNG', 'INI20030780': 'LIVER'}

In [4]:
data_d = '/home/lucytian/data/1_Single_Cell_PRS/2_cV2F/mvp_afr_0.9_0.01/20231221/406k_geno_v2_UKB_18PCs/fit_w_val/'

In [5]:
bcount_df = pd.read_csv('/home/lucytian/group/data/gwas_geno/blood_count.tsv.gz', sep='\t', compression='gzip', low_memory=False)
bchem_df = pd.read_csv('/home/lucytian/group/data/gwas_geno/blood_biochemistry.tsv.gz', sep='\t', compression='gzip', low_memory=False)
spiro_df = pd.read_csv('/home/lucytian/group/data/gwas_geno/spirometry.tsv.gz', sep='\t', compression='gzip', low_memory=False)

In [6]:
trait_category_dict = {'INI30120': bcount_df, 'INI50030700':bchem_df, 'INI20030780':bchem_df, 'INI1003063':spiro_df, 'INI10030620': bchem_df}

In [16]:
def generate_input(pop):
    tissue_trait_dict = {'INI30120': 'BLOOD', 'INI50030700': 'KIDNEY', 'INI1003063': 'LUNG', 'INI20030780': 'LIVER'}
    for tr in trait:
        ti = tissue_trait_dict[tr]
        base = trait_category_dict[tr][['#FID', 'IID', tr, 'population', 'split']]
        base = base[(base['population'] == pop) & (base['split'] == 'test')]
        if tr == "INI20030780":
            df_base = pd.read_csv(f'{data_d}/all/{tr}/exclude_APOE/snpnet.sscore.zst', sep='\t', compression='zstd')
            df_base = df_base.rename(columns = {'exclude_APOE_SUM': tr+'_baseline'})
        else:
            df_base = pd.read_csv(f'{data_d}/all/{tr}/snpnet.sscore.zst', sep='\t', compression='zstd')
            df_base = df_base.rename(columns = {tr+'_SUM': tr+'_baseline'})
        base = base.merge(df_base[['IID', tr+'_baseline']], on='IID')
        if tr == "INI20030780":
            df = pd.read_csv(data_d + tissue_trait_dict[tr] + '/' + tr + '/exclude_APOE/snpnet.sscore.zst', sep='\t', compression='zstd')
            df = df.rename(columns = {'exclude_APOE_SUM': tr+'_'+ti})
        else:
            df = pd.read_csv(data_d + tissue_trait_dict[tr] + '/' + tr + '/snpnet.sscore.zst', sep='\t', compression='zstd')
            df = df.rename(columns = {tr+'_SUM': tr+'_'+ti})
        base = base.merge(df[['IID', tr+'_'+ti]], on='IID')
        base = base.dropna()
        base = base.drop(columns=['population', 'split'])
        base.iloc[:, 2:].to_csv(f'by_trait_{tr}_{pop}.tsv', sep='\t', index=False)   

In [17]:
generate_input('Afr')

In [21]:
dfs = []
for tr in trait:
    r2redux_df = pd.read_csv(f'by_trait_{tr}_Afr_output.csv')
    dfs.append(r2redux_df)

In [22]:
pd.concat(dfs)

,rsq1,rsq2,var1,var2,var_diff,r2_based_p,r2_based_p_one_tail,mean_diff,upper_diff,lower_diff,p.nested,p.nonnested,p.LRT,Phenotype
0,0.007271,0.005921,0.000026,0.000022,0.000005,0.528988,0.264494,0.001350,0.005555,-0.002854,0.211274,0.528988,0.210380,INI30120
0,0.017899,0.012875,0.000062,0.000046,0.000009,0.093372,0.046686,0.005024,0.010892,-0.000845,0.016447,0.093372,0.016156,INI50030700
0,0.006667,0.005995,0.000026,0.000024,0.000005,0.754970,0.377485,0.000672,0.004891,-0.003547,0.394440,0.754970,0.393255,INI1003063
0,0.076134,0.075835,0.000231,0.000230,0.000023,0.949981,0.474990,0.000299,0.009652,-0.009053,0.560731,0.949981,0.545164,INI20030780
